# Residual stationarity

ADF and KPSS on the regression residual, read against a random-walk control that was detrended in exactly the same way.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.features` and `xauusd_quant.research` and read back from
`results/regression_research/<timeframe>/window_<N>/`. Generate those first:

```
xq regression-research --timeframe 1h --window 128
xq regression-research --timeframe 5m --all-windows
xq regression-research --all
```

No trading rule is defined or evaluated anywhere in this notebook. No
transaction costs are applied. Nothing here is a signal.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.features import load_regression_config

config = load_config(ROOT / "config" / "data.yaml")
regression = load_regression_config(ROOT / "config" / "regression.yaml")

TIMEFRAME = "1h"          # change and re-run
WINDOW = 128              # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"transform  : {regression.regression.price_transform} of "
      f"{regression.regression.price_source} {regression.regression.price_column}")
print(f"windows    : {list(regression.regression.windows)}")
print(f"fingerprint: {regression.fingerprint()}")

In [ ]:
import json

results = regression.results_dir(TIMEFRAME, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} window={WINDOW}. Run:  "
        f"xq regression-research --timeframe {TIMEFRAME} --window {WINDOW}"
    )

def table(name):
    """Load a previously-generated regression table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
provenance = summary["provenance"]
model = summary.get("model_summary", {})

print(f"generated  : {provenance['generated_utc']}")
print(f"window     : {provenance['regression_window']} bars on {provenance['timeframe']}")
print(f"valid fits : {provenance['observations']:,}")
print(f"span       : {provenance['requested_start']} -> {provenance['requested_end']}")
print(f"git        : {provenance['git']}")
for warning in summary.get("warnings", []):
    print(f"WARNING: {warning}")

## Why "the residual is stationary" is not a finding

Subtracting a locally fitted trend from any series - including one with no
mean reversion whatsoever - produces something that looks bounded and
zero-centred. A unit-root test run on it will usually reject the unit root.
That rejection is a property of the **detrending**, not of the market.

So every stationarity result here is paired with a control: the same rolling
regression, the same window, applied to a simulated random walk. If the
control gives the same verdict, the verdict carries no information about
XAUUSD.

The two tests have opposite nulls and are never collapsed into one number:

- **ADF** H0 = unit root (non-stationary). A small p rejects the unit root.
- **KPSS** H0 = stationary. A small p rejects stationarity.

In [ ]:
stationarity = json.loads((results / "stationarity.json").read_text(encoding="utf-8"))

for name in ("adf", "kpss"):
    test = stationarity.get(name)
    if test:
        print(f"{name.upper():5} statistic={test['statistic']:10.4f}  "
              f"p={test['p_value']:.4g}  rejects_at_5pct={test['reject_at_5pct']}")

print(f"\nresidual verdict : {stationarity['verdict']}")
print(f"control verdict  : {stationarity['control_verdict']}")
print(f"control matches  : {stationarity['control_matches']}")

## The control, side by side

`control_matches = True` is the outcome to take seriously: it means a pure
random walk, detrended identically, produced the same verdict as the real
residual.

In [ ]:
for name in ("control_adf", "control_kpss"):
    test = stationarity.get(name)
    if test:
        print(f"{name:14} statistic={test['statistic']:10.4f}  p={test['p_value']:.4g}")

print()
print(stationarity["interpretation"])
print()
for note in stationarity.get("notes", []):
    print(f"- {note}")

## Segmented

A verdict that holds in every segment is more credible than one that appears
only in the full sample. A verdict that changes between segments is evidence of
structural change - or of a sample too short to say anything.

In [ ]:
table('stationarity_segments')

## What this does and does not license

Stationarity is a statement about the distribution of a series. It is **not** a
statement about profitability, and it is not cointegration: there is no second
asset here, and no economic relationship being tested. A single price series
detrended against its own recent history is one series, not a spread.